# AI Fraud Detection — Robustness Improvement Run

Complete **FULL_MODE** Colab workflow. Phases 1–4 are unchanged. Phase 5 uses a larger, multi-attack, weighted adversarial-training pool created only from training rows. Results are measured, never assumed.

> Run cells in numerical order. The notebook saves improved artifacts separately and never overwrites the baseline model.


# AI Fraud Detection and Adversarial Robustness — FULL MODE

Complete Google Colab workflow for Phases 1–5 using the full PaySim dataset. Phases 6–8 are provided through the Streamlit dashboard after experiment artifacts are saved.

> Run cells in numerical order. Cell 1 may restart the runtime once after dependency installation; after reconnecting, rerun Cell 1 and continue from Cell 2. Full-data training, SHAP, and adversarial attack cells require substantially more RAM and time than DEVELOPMENT_MODE. Outputs are saved separately under `outputs_full_mode` so development artifacts are not overwritten.


### Cell 1 — Environment / Imports

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import signal
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/San-Yamin/AI_Fraud_Adversarial.git'
COLAB_PROJECT_ROOT = Path('/content/AI_Fraud_Adversarial')
project_candidates = [Path.cwd(), COLAB_PROJECT_ROOT]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'requirements-colab.txt').is_file()),
    None,
)
if PROJECT_ROOT is None:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(COLAB_PROJECT_ROOT)], check=True)
    PROJECT_ROOT = COLAB_PROJECT_ROOT
elif (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError(f'Project src folder not found under {PROJECT_ROOT}')

print('Project root:', PROJECT_ROOT)
%pip install -q --upgrade -r requirements-colab.txt

loaded_numpy = sys.modules.get('numpy')
installed_numpy_version = importlib.metadata.version('numpy')
if loaded_numpy is not None and loaded_numpy.__version__ != installed_numpy_version:
    print(
        f'NumPy changed from {loaded_numpy.__version__} to {installed_numpy_version}. '
        'Restarting the Colab runtime once; after reconnecting, rerun Cell 1.',
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGTERM)

import joblib
import matplotlib.pyplot as plt
import pandas as pd

print('src import path ready:', (PROJECT_ROOT / 'src').is_dir())


### Cell 2 — Google Drive Mount

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    raise RuntimeError('Google Drive is not mounted. Reconnect and rerun Cell 2.')
print('Google Drive ready:', DRIVE_ROOT)


### Cell 3 — Configuration

In [ ]:
# FULL MODE configuration. Development-mode artifacts remain untouched.
DEVELOPMENT_MODE = False
RUN_MODE = 'FULL_MODE'
RANDOM_SEED = 42

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/AI_Fraud_Adversarial')
PREFERRED_DATASET_PATH = DRIVE_PROJECT_DIR / 'data' / 'paysim.csv'

# Keep the expected path configurable, while helping with common PaySim filenames.
if PREFERRED_DATASET_PATH.is_file():
    DATASET_PATH = PREFERRED_DATASET_PATH
else:
    data_directory = DRIVE_PROJECT_DIR / 'data'
    discovered_csv_files = sorted(data_directory.glob('*.csv')) if data_directory.is_dir() else []
    paysim_candidates = [
        path for path in discovered_csv_files
        if 'paysim' in path.name.lower() or '20174392719' in path.name
    ]
    if len(paysim_candidates) == 1:
        DATASET_PATH = paysim_candidates[0]
        print('Using discovered PaySim file:', DATASET_PATH)
    else:
        available = [str(path) for path in discovered_csv_files]
        raise FileNotFoundError(
            'PaySim CSV was not found at the expected path. Upload/rename it to '
            f'{PREFERRED_DATASET_PATH}. CSV files currently in the data folder: {available}'
        )

OUTPUT_DIR = DRIVE_PROJECT_DIR / 'outputs_full_mode'
FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase1'
MODELS_DIR = OUTPUT_DIR / 'models'
METRICS_DIR = OUTPUT_DIR / 'metrics'
for directory in (FIGURES_DIR, MODELS_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Ensure every reusable src module sees the same Full Mode paths and settings.
os.environ['RUN_MODE'] = RUN_MODE
os.environ['RANDOM_SEED'] = str(RANDOM_SEED)
os.environ['PAYSIM_DATASET_PATH'] = str(DATASET_PATH)
os.environ['OUTPUT_DIR'] = str(OUTPUT_DIR)

# Safe when Cell 3 is rerun in a live notebook session.
if 'src.config' in sys.modules:
    import importlib
    import src.config
    importlib.reload(src.config)

assert RUN_MODE == 'FULL_MODE'
assert DATASET_PATH.is_file()
print({
    'run_mode': RUN_MODE,
    'dataset': str(DATASET_PATH),
    'dataset_size_gb': round(DATASET_PATH.stat().st_size / 1e9, 2),
    'outputs': str(OUTPUT_DIR),
})


### Cell 4 — Dataset Check

In [ ]:
if Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

checks = {
    'project_src_exists': (PROJECT_ROOT / 'src').is_dir(),
    'drive_mounted': Path('/content/drive/MyDrive').is_dir(),
    'dataset_exists': DATASET_PATH.is_file(),
    'full_mode': RUN_MODE == 'FULL_MODE',
}
print(checks)
if not all(checks.values()):
    raise RuntimeError('Setup check failed. Rerun Cells 1, 2, and 3 in order.')
print(f'PaySim found: {DATASET_PATH} ({DATASET_PATH.stat().st_size / 1e9:.2f} GB)')


### Cell 5 — Load PaySim

In [ ]:
from src.data_loader import load_paysim
data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
print(f'Loaded {len(data):,} rows in {RUN_MODE}.')

### Cell 6 — Inspect Dataset

In [ ]:
from src.data_loader import summarize_paysim
summary = summarize_paysim(data)
print('Shape:', summary['shape'])
print('Columns:', summary['columns'])
print('Missing values:', summary['missing_values'])
print('Transaction types:', summary['transaction_types'])
print('Class counts:', summary['class_counts'])
print(f"Fraud percentage: {summary['fraud_percentage']:.6f}%")

### Cell 7 — Preprocess and Engineer Features

In [ ]:
from src.preprocessing import prepare_features_and_target
X, y = prepare_features_and_target(data)
print('Raw model inputs:', X.columns.tolist())
print('Target counts:', y.value_counts().sort_index().to_dict())

### Cell 8 — Train/Test Split

In [ ]:
from src.preprocessing import fit_transform_train_test
from src.train import stratified_train_test_split
X_train_raw, X_test_raw, y_train, y_test = stratified_train_test_split(
    X, y, random_state=RANDOM_SEED
)
X_train, X_test, preprocessor = fit_transform_train_test(X_train_raw, X_test_raw)
feature_names = X_train.columns.tolist()
print('Training counts:', y_train.value_counts().sort_index().to_dict())
print('Untouched test counts:', y_test.value_counts().sort_index().to_dict())
print('Encoded features:', feature_names)

### Cell 9 — SMOTE

In [ ]:
from src.config import SMOTE_SAMPLING_STRATEGY
from src.preprocessing import resample_training_data
print('Before SMOTE:', y_train.value_counts().sort_index().to_dict())
X_train_smote, y_train_smote = resample_training_data(
    X_train, y_train, random_state=RANDOM_SEED,
    sampling_strategy=SMOTE_SAMPLING_STRATEGY
)
print('After SMOTE:', pd.Series(y_train_smote).value_counts().sort_index().to_dict())
print('Test data was not passed to SMOTE.')

### Cell 10 — Train Baseline Model

In [ ]:
from src.train import train_baseline_model
baseline_model = train_baseline_model(X_train_smote, y_train_smote)
print('Baseline XGBoost training complete.')

### Cell 11 — Evaluate Baseline

In [ ]:
from src.evaluate import evaluate_binary_classifier
from src.visualization import (
    plot_confusion_matrix, plot_original_class_distribution,
    plot_precision_recall, plot_smote_distributions,
)
evaluation = evaluate_binary_classifier(baseline_model, X_test, y_test)
print(json.dumps(evaluation['metrics'], indent=2))
print('Fraud Recall (primary concern):', evaluation['metrics']['recall'])
print(pd.DataFrame(evaluation['classification_report']).transpose())
plot_original_class_distribution(y, FIGURES_DIR)
plot_smote_distributions(y_train, y_train_smote, FIGURES_DIR)
plot_confusion_matrix(evaluation['confusion_matrix'], FIGURES_DIR)
plot_precision_recall(evaluation, FIGURES_DIR)
plt.show()

### Cell 12 — Save Artifacts

In [ ]:
from src.evaluate import save_metrics
baseline_model_path = MODELS_DIR / 'baseline_model.joblib'
feature_names_path = MODELS_DIR / 'feature_names.joblib'
preprocessor_path = MODELS_DIR / 'baseline_preprocessor.joblib'
metrics_path = METRICS_DIR / 'phase1_baseline_metrics.json'
joblib.dump(baseline_model, baseline_model_path)
joblib.dump(feature_names, feature_names_path)
joblib.dump(preprocessor, preprocessor_path)
save_metrics(evaluation, metrics_path)
print('Saved:', baseline_model_path, feature_names_path, preprocessor_path, metrics_path, sep='\n- ')

# PHASE 2 — SHAP Explainability
Reuse the trained Phase 1 baseline. Do not retrain it.

### Cell 13 — Load Phase 1 Artifacts

In [ ]:
from src.explainability import load_phase1_artifacts
baseline_model, saved_preprocessor, saved_feature_names = load_phase1_artifacts(
    MODELS_DIR / 'baseline_model.joblib',
    MODELS_DIR / 'baseline_preprocessor.joblib',
    MODELS_DIR / 'feature_names.joblib',
)
print(f'Loaded baseline model with {len(saved_feature_names)} encoded features.')
print('The baseline model was loaded, not retrained.')

### Cell 14 — Prepare SHAP Evaluation Sample

In [ ]:
from src.explainability import prepare_shap_evaluation_sample, transform_with_saved_preprocessor
# Reuse the live Phase 1 test split. Reconstruct it deterministically after a restart.
if 'X_test' not in globals() or 'y_test' not in globals():
    from src.data_loader import load_paysim
    from src.preprocessing import prepare_features_and_target
    from src.train import stratified_train_test_split
    reconstructed_data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
    reconstructed_X, reconstructed_y = prepare_features_and_target(reconstructed_data)
    _, X_test_raw, _, y_test = stratified_train_test_split(
        reconstructed_X, reconstructed_y, random_state=RANDOM_SEED
    )
    X_test = transform_with_saved_preprocessor(
        saved_preprocessor, X_test_raw, saved_feature_names
    )
else:
    if X_test.columns.tolist() != saved_feature_names:
        raise ValueError('Live test features do not match saved Phase 1 feature names.')
X_shap, y_shap = prepare_shap_evaluation_sample(
    X_test, y_test, random_state=RANDOM_SEED
)
print('SHAP sample shape:', X_shap.shape)
print('SHAP sample labels:', y_shap.value_counts().sort_index().to_dict())
print('This class-aware held-out sample is for explanation, not prevalence estimation.')

### Cell 15 — Create SHAP Explainer

In [ ]:
from src.explainability import create_tree_explainer, compute_shap_values
shap_explainer = create_tree_explainer(baseline_model)
shap_values = compute_shap_values(shap_explainer, X_shap)
print('Computed Tree SHAP values:', shap_values.values.shape)
print('Positive SHAP values push the raw model margin toward fraud; negative values push toward legitimate.')

### Cell 16 — Global SHAP Feature Importance

In [ ]:
from src.explainability import rank_global_importance
global_importance = rank_global_importance(shap_values)
display(global_importance)
import shap
shap.plots.bar(shap_values, max_display=15)

### Cell 17 — SHAP Summary Plot

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15)

### Cell 18 — Explain Correctly Detected Fraud

In [ ]:
from src.explainability import build_interpretation, explain_selected_row, select_correct_examples
selected_examples = select_correct_examples(
    baseline_model, X_test, y_test, random_state=RANDOM_SEED
)
fraud_explanation, fraud_table = explain_selected_row(
    shap_explainer, X_test, selected_examples['fraud']
)
fraud_interpretation = build_interpretation(
    selected_examples['fraud'], fraud_table
)
print(json.dumps(fraud_interpretation, indent=2, default=str))
display(fraud_table.head(15))
shap.plots.waterfall(fraud_explanation, max_display=15)

### Cell 19 — Explain Correctly Detected Legitimate Transaction

In [ ]:
legitimate_explanation, legitimate_table = explain_selected_row(
    shap_explainer, X_test, selected_examples['legitimate']
)
legitimate_interpretation = build_interpretation(
    selected_examples['legitimate'], legitimate_table
)
print(json.dumps(legitimate_interpretation, indent=2, default=str))
display(legitimate_table.head(15))
shap.plots.waterfall(legitimate_explanation, max_display=15)

### Cell 20 — Save SHAP Outputs and Interpretation

In [ ]:
from src.explainability import save_global_plots, save_phase2_outputs, save_waterfall_plot
SHAP_OUTPUT_DIR = OUTPUT_DIR / 'shap'
save_global_plots(shap_values, SHAP_OUTPUT_DIR, max_display=15)
save_waterfall_plot(
    fraud_explanation, SHAP_OUTPUT_DIR / 'correct_fraud_waterfall.png', max_display=15
)
save_waterfall_plot(
    legitimate_explanation, SHAP_OUTPUT_DIR / 'correct_legitimate_waterfall.png', max_display=15
)
saved_shap_paths = save_phase2_outputs(
    SHAP_OUTPUT_DIR,
    global_importance,
    {'fraud': fraud_table, 'legitimate': legitimate_table},
    {'fraud': fraud_interpretation, 'legitimate': legitimate_interpretation},
    y_shap,
)
print('Saved Phase 2 outputs:')
for path in sorted(SHAP_OUTPUT_DIR.iterdir()):
    print('-', path)

# PHASE 3 — Adversarial Evasion Attack
Targeted, decision-based HopSkipJump on correctly detected held-out fraud only. Generated test samples are evaluation-only and must never be used for training.

### Cell 21 — ART Setup and Compatibility Check

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec('art') is None:
    print('ART is missing; installing adversarial-robustness-toolbox from requirements-colab.txt...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import art
from art.attacks.evasion import HopSkipJump
from art.estimators.classification import BlackBoxClassifier, XGBoostClassifier
from src.attack import verify_art_compatibility
art_compatibility = verify_art_compatibility(baseline_model, saved_feature_names)
print(json.dumps(art_compatibility, indent=2))
print('Available ART wrappers:', BlackBoxClassifier.__name__, XGBoostClassifier.__name__)
print('Selected attack:', HopSkipJump.__name__)
print('Reason: HSJ needs predictions/decisions, not gradients, and the black-box adapter enforces valid feature dependencies.')

### Cell 22 — Build Feature Threat Model

In [ ]:
from src.attack import build_feature_threat_model
from src.config import ATTACK_RELATIVE_BOUND
feature_threat_model = build_feature_threat_model(
    X_test, relative_bound=ATTACK_RELATIVE_BOUND
)
display(feature_threat_model)
print('Direct monetary changes are bounded and non-negative.')
print('Engineered values are recomputed; step and one-hot transaction type are protected.')

### Cell 23 — Select Correctly Detected Fraud Test Samples

In [ ]:
from src.attack import select_correctly_detected_test_fraud
from src.config import ATTACK_SAMPLE_SIZE
X_attack_clean, y_attack, attack_population = select_correctly_detected_test_fraud(
    baseline_model, X_test, y_test, sample_size=ATTACK_SAMPLE_SIZE,
    random_state=RANDOM_SEED,
)
print(json.dumps(attack_population, indent=2))
print('Selected source test indices:', X_attack_clean.index.tolist())
assert y_attack.eq(1).all()
assert (baseline_model.predict(X_attack_clean) == 1).all()

### Cell 24 — Configure Adversarial Attack

In [ ]:
from src.config import (ATTACK_INIT_EVAL, ATTACK_INIT_SIZE, ATTACK_MAX_EVAL, ATTACK_MAX_ITER)
attack_configuration = {
    'attack': 'Targeted HopSkipJump',
    'target_class': 0,
    'relative_monetary_bound': ATTACK_RELATIVE_BOUND,
    'max_iter': ATTACK_MAX_ITER,
    'max_eval': ATTACK_MAX_EVAL,
    'init_eval': ATTACK_INIT_EVAL,
    'init_size': ATTACK_INIT_SIZE,
    'random_seed': RANDOM_SEED,
    'evaluation_only': True,
}
print(json.dumps(attack_configuration, indent=2))

### Cell 25 — Generate Adversarial TEST Samples

In [ ]:
from src.attack import generate_constrained_hopskipjump
X_attack_adversarial, attack_execution = generate_constrained_hopskipjump(
    baseline_model, X_attack_clean,
    relative_bound=ATTACK_RELATIVE_BOUND,
    max_iter=ATTACK_MAX_ITER, max_eval=ATTACK_MAX_EVAL,
    init_eval=ATTACK_INIT_EVAL, init_size=ATTACK_INIT_SIZE,
    random_state=RANDOM_SEED, verbose=True,
)
print(json.dumps(attack_execution, indent=2))
print('Generated adversarial test matrix:', X_attack_adversarial.shape)
print('Original X_test remains unchanged; adversarial rows are stored separately.')

### Cell 26 — Evaluate Recall Under Attack

In [ ]:
from src.evaluate import evaluate_adversarial_evasion
phase3_metrics, attacked_samples = evaluate_adversarial_evasion(
    baseline_model, X_attack_clean, X_attack_adversarial,
    full_test_clean_fraud_recall=attack_population['full_test_clean_fraud_recall'],
    attack_metadata=attack_execution,
)
print('Full-test clean fraud recall:', phase3_metrics['full_test_clean_fraud_recall'])
print('Selected-sample clean recall:', phase3_metrics['attacked_sample_clean_recall'])
print('Recall under attack:', phase3_metrics['recall_under_attack'])
print('Recall drop:', phase3_metrics['recall_drop_on_attacked_sample'])

### Cell 27 — Calculate Attack Success Rate

In [ ]:
from src.visualization import plot_attack_perturbations, plot_attack_probabilities
PHASE3_FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase3'
print('Attack success definition:', phase3_metrics['attack_success_definition'])
print('Successful evasions:', phase3_metrics['successful_evasions'])
print('Number attacked:', phase3_metrics['number_attacked'])
print('Attack success/evasion rate:', phase3_metrics['attack_success_rate'])
print('Average L2 perturbation:', phase3_metrics['average_l2_perturbation'])
print('Maximum L2 perturbation:', phase3_metrics['max_l2_perturbation'])
display(attacked_samples)
plot_attack_probabilities(attacked_samples, PHASE3_FIGURES_DIR)
plot_attack_perturbations(attacked_samples, PHASE3_FIGURES_DIR)
plt.show()

### Cell 28 — Save Phase 3 Attack Outputs

In [ ]:
from src.evaluate import save_attack_outputs
phase3_metrics_path = METRICS_DIR / 'phase3_attack_metrics.json'
phase3_samples_path = METRICS_DIR / 'phase3_attacked_samples.csv'
phase3_threat_model_path = METRICS_DIR / 'phase3_feature_constraints.csv'
save_attack_outputs(
    phase3_metrics, attacked_samples, phase3_metrics_path, phase3_samples_path
)
feature_threat_model.to_csv(phase3_threat_model_path, index=False)
print('Saved Phase 3 outputs:')
for path in (phase3_metrics_path, phase3_samples_path, phase3_threat_model_path):
    print('-', path)
for path in sorted(PHASE3_FIGURES_DIR.iterdir()):
    print('-', path)
print('Evaluation-only adversarial test samples were not saved as training data.')

# PHASE 4 — Multiple Attack Comparison
Compare three compatible ART attacks on the same constrained, evaluation-only held-out fraud population. Do not train on these samples.

### Cell 29 — Phase 4 Setup

In [ ]:
from src import config as project_config
print('Phase 4 config loaded from:', project_config.__file__)
ATTACK_INIT_EVAL = getattr(project_config, 'ATTACK_INIT_EVAL', 50)
ATTACK_INIT_SIZE = getattr(project_config, 'ATTACK_INIT_SIZE', 30)
ATTACK_MAX_EVAL = getattr(project_config, 'ATTACK_MAX_EVAL', 500)
ATTACK_MAX_ITER = getattr(project_config, 'ATTACK_MAX_ITER', 10)
ATTACK_RELATIVE_BOUND = getattr(project_config, 'ATTACK_RELATIVE_BOUND', 0.10)
PHASE4_ATTACK_SAMPLE_SIZE = getattr(
    project_config, 'PHASE4_ATTACK_SAMPLE_SIZE',
    getattr(project_config, 'ATTACK_SAMPLE_SIZE', 20),
)
BOUNDARY_MAX_ITER = getattr(project_config, 'BOUNDARY_MAX_ITER', 50)
BOUNDARY_NUM_TRIAL = getattr(project_config, 'BOUNDARY_NUM_TRIAL', 10)
BOUNDARY_SAMPLE_SIZE = getattr(project_config, 'BOUNDARY_SAMPLE_SIZE', 10)
ZOO_MAX_ITER = getattr(project_config, 'ZOO_MAX_ITER', 20)
ZOO_LEARNING_RATE = getattr(project_config, 'ZOO_LEARNING_RATE', 0.05)
ZOO_NB_PARALLEL = getattr(project_config, 'ZOO_NB_PARALLEL', 5)
phase4_attack_parameters = {
    'HopSkipJump': {
        'max_iter': ATTACK_MAX_ITER, 'max_eval': ATTACK_MAX_EVAL,
        'init_eval': ATTACK_INIT_EVAL, 'init_size': ATTACK_INIT_SIZE,
    },
    'BoundaryAttack': {
        'max_iter': BOUNDARY_MAX_ITER, 'num_trial': BOUNDARY_NUM_TRIAL,
        'sample_size': BOUNDARY_SAMPLE_SIZE, 'init_size': ATTACK_INIT_SIZE,
    },
    'ZooAttack': {
        'max_iter': ZOO_MAX_ITER, 'learning_rate': ZOO_LEARNING_RATE,
        'nb_parallel': ZOO_NB_PARALLEL,
    },
}
phase4_results = {}
phase4_sample_results = {}
print(json.dumps(phase4_attack_parameters, indent=2))

### Cell 30 — Verify Compatible Attacks

In [ ]:
from src.attack import verify_art_compatibility, verify_comparison_attacks
phase4_model_compatibility = verify_art_compatibility(
    baseline_model, saved_feature_names
)
phase4_attack_compatibility = verify_comparison_attacks()
print(json.dumps(phase4_model_compatibility, indent=2))
display(phase4_attack_compatibility)
assert phase4_attack_compatibility['compatible'].all()
assert not phase4_attack_compatibility['gradient_required'].any()

### Cell 31 — Prepare Common Attack Population

In [ ]:
from src.attack import select_correctly_detected_test_fraud
X_phase4_clean, y_phase4_attack, phase4_population = select_correctly_detected_test_fraud(
    baseline_model, X_test, y_test, sample_size=PHASE4_ATTACK_SAMPLE_SIZE,
    random_state=RANDOM_SEED,
)
print(json.dumps(phase4_population, indent=2))
print('Common source test indices:', X_phase4_clean.index.tolist())
assert y_phase4_attack.eq(1).all()
assert (baseline_model.predict(X_phase4_clean) == 1).all()

### Cell 32 — Run Attack 1

In [ ]:
from src.attack_comparison import run_comparison_attack
X_phase4_hsj, phase4_results['HopSkipJump'], phase4_sample_results['HopSkipJump'] = run_comparison_attack(
    'HopSkipJump', baseline_model, X_phase4_clean,
    full_test_clean_fraud_recall=phase4_population['full_test_clean_fraud_recall'],
    parameters=phase4_attack_parameters['HopSkipJump'],
    relative_bound=ATTACK_RELATIVE_BOUND, random_state=RANDOM_SEED, verbose=True,
)
print(json.dumps(phase4_results['HopSkipJump'], indent=2, default=str))

### Cell 33 — Run Attack 2

In [ ]:
X_phase4_boundary, phase4_results['BoundaryAttack'], phase4_sample_results['BoundaryAttack'] = run_comparison_attack(
    'BoundaryAttack', baseline_model, X_phase4_clean,
    full_test_clean_fraud_recall=phase4_population['full_test_clean_fraud_recall'],
    parameters=phase4_attack_parameters['BoundaryAttack'],
    relative_bound=ATTACK_RELATIVE_BOUND, random_state=RANDOM_SEED, verbose=True,
)
print(json.dumps(phase4_results['BoundaryAttack'], indent=2, default=str))

### Cell 34 — Run Attack 3 if Practical

In [ ]:
X_phase4_zoo, phase4_results['ZooAttack'], phase4_sample_results['ZooAttack'] = run_comparison_attack(
    'ZooAttack', baseline_model, X_phase4_clean,
    full_test_clean_fraud_recall=phase4_population['full_test_clean_fraud_recall'],
    parameters=phase4_attack_parameters['ZooAttack'],
    relative_bound=ATTACK_RELATIVE_BOUND, random_state=RANDOM_SEED, verbose=True,
)
print(json.dumps(phase4_results['ZooAttack'], indent=2, default=str))

### Cell 35 — Build Comparison Table

In [ ]:
from src.attack_comparison import build_comparison_table, identify_comparison_findings
phase4_comparison = build_comparison_table(phase4_results)
phase4_findings = identify_comparison_findings(phase4_comparison)
display(phase4_comparison)
print(json.dumps(phase4_findings, indent=2))
print('Rankings apply only to this population, constraint set, and configured budgets.')

### Cell 36 — Plot Attack Comparison

In [ ]:
from src.visualization import plot_attack_comparison
PHASE4_FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase4'
phase4_figures = plot_attack_comparison(phase4_comparison, PHASE4_FIGURES_DIR)
plt.show()

### Cell 37 — Save Phase 4 Outputs

In [ ]:
from src.attack_comparison import save_comparison_outputs
phase4_csv_path = METRICS_DIR / 'phase4_attack_comparison.csv'
phase4_json_path = METRICS_DIR / 'phase4_attack_comparison.json'
save_comparison_outputs(
    phase4_comparison, phase4_findings, phase4_attack_compatibility,
    phase4_csv_path, phase4_json_path,
)
print('Saved Phase 4 outputs:', phase4_csv_path, phase4_json_path, sep='\n- ')
for path in sorted(PHASE4_FIGURES_DIR.iterdir()):
    print('-', path)
print('All adversarial samples remain evaluation-only and were not used for training.')

# PHASE 5 — Model Hardening

### Cell 38 — Phase 5 Setup

In [ ]:
from src import config as project_config
from src.explainability import load_phase1_artifacts, transform_with_saved_preprocessor
from src.data_loader import load_paysim
from src.preprocessing import prepare_features_and_target, resample_training_data
from src.train import stratified_train_test_split

baseline_model, saved_preprocessor, saved_feature_names = load_phase1_artifacts(
    MODELS_DIR / 'baseline_model.joblib',
    MODELS_DIR / 'baseline_preprocessor.joblib',
    MODELS_DIR / 'feature_names.joblib',
)

# Reconstruct the exact FULL_MODE split if this session was restarted.
if not all(name in globals() for name in ('X_train', 'X_test', 'y_train', 'y_test')):
    phase5_data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
    phase5_X_raw, phase5_y = prepare_features_and_target(phase5_data)
    X_train_raw, X_test_raw, y_train, y_test = stratified_train_test_split(
        phase5_X_raw, phase5_y, random_state=RANDOM_SEED
    )
    X_train = transform_with_saved_preprocessor(
        saved_preprocessor, X_train_raw, saved_feature_names
    )
    X_test = transform_with_saved_preprocessor(
        saved_preprocessor, X_test_raw, saved_feature_names
    )
if not all(name in globals() for name in ('X_train_smote', 'y_train_smote')):
    X_train_smote, y_train_smote = resample_training_data(
        X_train, y_train, random_state=RANDOM_SEED,
        sampling_strategy=project_config.SMOTE_SAMPLING_STRATEGY,
    )

ROBUST_TRAIN_SAMPLES = int(os.getenv(
    'ROBUST_HARDENING_TRAIN_SAMPLE_SIZE', '60' if DEVELOPMENT_MODE else '200'
))
ROBUST_TEST_SAMPLES = int(os.getenv(
    'ROBUST_HARDENING_TEST_SAMPLE_SIZE', '20' if DEVELOPMENT_MODE else '50'
))
ADVERSARIAL_WEIGHT = float(os.getenv(
    'ROBUST_HARDENING_ADVERSARIAL_WEIGHT', '50.0'
))
PHASE5_IMPROVED_FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase5_robustness_improved'
PHASE5_IMPROVED_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
phase5_attack_parameters = {
    'HopSkipJump': {
        'max_iter': project_config.ATTACK_MAX_ITER,
        'max_eval': project_config.ATTACK_MAX_EVAL,
        'init_eval': project_config.ATTACK_INIT_EVAL,
        'init_size': project_config.ATTACK_INIT_SIZE,
    },
    'BoundaryAttack': {
        'max_iter': project_config.BOUNDARY_MAX_ITER,
        'num_trial': project_config.BOUNDARY_NUM_TRIAL,
        'sample_size': project_config.BOUNDARY_SAMPLE_SIZE,
        'init_size': project_config.ATTACK_INIT_SIZE,
    },
    'ZooAttack': {
        'max_iter': project_config.ZOO_MAX_ITER,
        'learning_rate': project_config.ZOO_LEARNING_RATE,
        'nb_parallel': project_config.ZOO_NB_PARALLEL,
    },
}
assert RUN_MODE == 'FULL_MODE'
assert X_train.index.intersection(X_test.index).empty
assert X_train.columns.tolist() == saved_feature_names == X_test.columns.tolist()
print({'mode': RUN_MODE, 'train_attack_sources': ROBUST_TRAIN_SAMPLES,
       'fresh_test_attack_samples': ROBUST_TEST_SAMPLES,
       'adversarial_weight': ADVERSARIAL_WEIGHT})
print('Baseline remains unchanged. No Phase 3/4 attacked test artifact is loaded.')


### Cell 39 — Select Adversarial TRAIN Source Samples

In [ ]:
from src.hardening import select_adversarial_training_sources
X_hardening_source, y_hardening_source, phase5_training_source = select_adversarial_training_sources(
    baseline_model, X_train, y_train,
    sample_size=ROBUST_TRAIN_SAMPLES,
    random_state=RANDOM_SEED,
)
assert X_hardening_source.index.intersection(X_test.index).empty
assert y_hardening_source.eq(1).all()
print(json.dumps(phase5_training_source, indent=2))
print('Only correctly detected fraud from the original training split was selected.')


### Cell 40 — Generate Adversarial TRAIN Examples

In [ ]:
from src.attack import generate_constrained_comparison_attack, validate_adversarial_constraints
from src.hardening import keep_successful_training_evasions

# Use disjoint training-source rows: 40% HSJ, 40% Boundary, 20% ZOO.
attack_names = ['HopSkipJump', 'BoundaryAttack', 'ZooAttack']
counts = [int(len(X_hardening_source) * 0.4), int(len(X_hardening_source) * 0.4)]
counts.append(len(X_hardening_source) - sum(counts))
phase5_adversarial_parts = []
phase5_train_attack_execution = {}
phase5_successful_training_attacks = {}
start = 0
for attack_offset, (attack_name, count) in enumerate(zip(attack_names, counts)):
    clean_part = X_hardening_source.iloc[start:start + count]
    start += count
    generated_part, execution = generate_constrained_comparison_attack(
        attack_name, baseline_model, clean_part,
        phase5_attack_parameters[attack_name],
        relative_bound=project_config.ATTACK_RELATIVE_BOUND,
        random_state=RANDOM_SEED + attack_offset * 1000,
        verbose=True,
    )
    validate_adversarial_constraints(
        clean_part, generated_part,
        relative_bound=project_config.ATTACK_RELATIVE_BOUND,
    )
    successful_part, success_metadata = keep_successful_training_evasions(
        baseline_model, generated_part
    )
    phase5_adversarial_parts.append(successful_part)
    phase5_train_attack_execution[attack_name] = execution
    phase5_successful_training_attacks[attack_name] = success_metadata

X_adversarial_train = pd.concat(phase5_adversarial_parts, axis=0)
assert X_adversarial_train.index.intersection(X_test.index).empty
print('Successful adversarial TRAIN examples retained:', len(X_adversarial_train))
display(pd.DataFrame(phase5_successful_training_attacks).T)
print('These are real training-side evasions. No result was manually changed.')


### Cell 41 — Build Augmented Training Set

In [ ]:
from src.hardening import build_augmented_training_set
X_hardened_train, y_hardened_train, phase5_augmentation = build_augmented_training_set(
    X_train_smote, y_train_smote, X_adversarial_train,
    source_training_indices=X_adversarial_train.index,
    untouched_test_indices=X_test.index,
)
print(json.dumps(phase5_augmentation, indent=2))
print('Every appended adversarial row keeps fraud label = 1.')


### Cell 42 — Train Hardened Model

In [ ]:
from src.hardening import train_weighted_hardened_model
hardened_model, phase5_weighting = train_weighted_hardened_model(
    X_hardened_train, y_hardened_train,
    clean_training_rows=len(X_train_smote),
    adversarial_weight=ADVERSARIAL_WEIGHT,
    random_state=RANDOM_SEED,
)
assert hardened_model is not baseline_model
print(json.dumps(phase5_weighting, indent=2))
print('A new weighted hardened model was trained; baseline_model was not refitted.')


### Cell 43 — Evaluate Hardened Model on Clean Test Set

In [ ]:
from src.evaluate import evaluate_binary_classifier
hardened_evaluation = evaluate_binary_classifier(hardened_model, X_test, y_test)
baseline_evaluation_phase5 = evaluate_binary_classifier(baseline_model, X_test, y_test)
clean_comparison = pd.DataFrame([
    {'Model': 'Baseline', **baseline_evaluation_phase5['metrics']},
    {'Model': 'Robustness-improved hardened', **hardened_evaluation['metrics']},
])
display(clean_comparison)
print('Hardened confusion matrix:', hardened_evaluation['confusion_matrix'])
print('The untouched test set was evaluation-only and was never augmented or fitted.')


### Cell 44 — Run Fresh Attacks Against Hardened Model

In [ ]:
from src.hardening import select_common_correct_test_fraud
from src.attack_comparison import run_comparison_attack

X_phase5_attack, phase5_test_population = select_common_correct_test_fraud(
    baseline_model, hardened_model, X_test, y_test,
    sample_size=ROBUST_TEST_SAMPLES,
    random_state=RANDOM_SEED,
)
phase5_baseline_attacks = {}
phase5_hardened_attacks = {}
phase5_attack_samples = {'Baseline': {}, 'Hardened': {}}
for attack_name, parameters in phase5_attack_parameters.items():
    _, phase5_baseline_attacks[attack_name], phase5_attack_samples['Baseline'][attack_name] = run_comparison_attack(
        attack_name, baseline_model, X_phase5_attack,
        full_test_clean_fraud_recall=phase5_test_population['baseline_clean_fraud_recall'],
        parameters=parameters, relative_bound=project_config.ATTACK_RELATIVE_BOUND,
        random_state=RANDOM_SEED, verbose=True,
    )
    _, phase5_hardened_attacks[attack_name], phase5_attack_samples['Hardened'][attack_name] = run_comparison_attack(
        attack_name, hardened_model, X_phase5_attack,
        full_test_clean_fraud_recall=phase5_test_population['hardened_clean_fraud_recall'],
        parameters=parameters, relative_bound=project_config.ATTACK_RELATIVE_BOUND,
        random_state=RANDOM_SEED, verbose=True,
    )
print(json.dumps(phase5_test_population, indent=2))
print('Fresh adaptive attacks were generated separately against each model.')


### Cell 45 — Compare Baseline vs Hardened Robustness

In [ ]:
from src.hardening import build_hardening_comparison
from src.visualization import plot_hardening_comparison
phase5_comparison = build_hardening_comparison(
    baseline_evaluation_phase5['metrics'], hardened_evaluation['metrics'],
    phase5_baseline_attacks, phase5_hardened_attacks,
)
display(phase5_comparison)
phase5_figures = plot_hardening_comparison(
    phase5_comparison, hardened_evaluation['confusion_matrix'],
    PHASE5_IMPROVED_FIGURES_DIR,
)
plt.show()
improved = (phase5_comparison['Recall Recovery'] > 0) | (
    phase5_comparison['Reduction in Attack Success Rate'] > 0
)
print('Attacks showing measured robustness improvement:',
      phase5_comparison.loc[improved, 'Attack'].tolist())
print('Zero improvement is a valid result; this notebook never fabricates success.')


### Cell 46 — Save Phase 5 Outputs

In [ ]:
from src.hardening import save_phase5_outputs
hardened_model_path = MODELS_DIR / 'hardened_model_robustness_improved.joblib'
phase5_metrics_path = METRICS_DIR / 'phase5_robustness_improved_metrics.json'
phase5_comparison_path = METRICS_DIR / 'phase5_robustness_improved_comparison.csv'
phase5_methodology = {
    'training_source': phase5_training_source,
    'successful_training_attacks': phase5_successful_training_attacks,
    'training_attack_execution': phase5_train_attack_execution,
    'augmentation': phase5_augmentation,
    'sample_weighting': phase5_weighting,
    'fresh_test_attack_population': phase5_test_population,
    'fresh_attacks': list(phase5_attack_parameters),
    'phase3_or_phase4_adversarial_test_samples_used_for_training': False,
    'baseline_model_overwritten': False,
}
save_phase5_outputs(
    hardened_model, hardened_evaluation, phase5_comparison, phase5_methodology,
    hardened_model_path, phase5_metrics_path, phase5_comparison_path,
)
print('Saved separate robustness-improvement outputs:', hardened_model_path,
      phase5_metrics_path, phase5_comparison_path, sep='\n- ')
for output_path in sorted(PHASE5_IMPROVED_FIGURES_DIR.iterdir()):
    print('-', output_path)
print('Send the metrics JSON and comparison CSV for evidence-based review.')


# PHASE 6 — Streamlit Dashboard
Implemented in `app.py` and `src/dashboard_utils.py`. The dashboard reads the saved Phase 1–5 artifacts without retraining.

Colab launch commands:

```python
%env PAYSIM_DATASET_PATH=/content/drive/MyDrive/AI_Fraud_Adversarial/data/paysim.csv
%env OUTPUT_DIR=/content/drive/MyDrive/AI_Fraud_Adversarial/outputs_full_mode
!streamlit run app.py --server.port 8501 &>/content/streamlit.log &
from google.colab import output
output.serve_kernel_port_as_window(8501)
```

# PHASE 7 — Real-Time Transaction Simulation
Implemented as the **Real-Time Simulation** dashboard section using `src/realtime_simulation.py`.

It streams a small reproducible, class-aware PaySim sample through the saved preprocessor and hardened model. `isFraud` is used only for evaluation/display. This is a synthetic simulation, not a live banking feed.

Use the Phase 6 launch commands above, then select **Real-Time Simulation** in the Streamlit sidebar.

# PHASE 8 — Concept Drift
Implemented as the **Concept Drift** section in the Streamlit dashboard using `src/concept_drift.py`.

The unchanged hardened model is evaluated over fixed, equal-width PaySim `step` ranges. Feature drift uses bounded deterministic samples and the two-sample Kolmogorov–Smirnov statistic. `isFraud` is retained only for metrics.

Launch the dashboard with the Phase 6 commands above, select **Concept Drift**, and run the analysis. This is simulated PaySim temporal analysis, not evidence of production drift.

# FINAL — Results and Conclusions

The implementation workflow is complete. Use only the metrics and figures produced by this execution when writing conclusions; no result is hard-coded in the notebook.

Collect the following from Google Drive for the final report:

- Phase 1 baseline Precision, Recall, F1, PR-AUC, confusion matrix, and PR curve
- Phase 2 global SHAP ranking, beeswarm, and two local waterfall explanations
- Phase 3 constrained evasion metrics and perturbation figures
- Phase 4 attack-comparison CSV/JSON and four comparison figures
- Phase 5 clean and adversarial baseline-versus-hardened results
- Phase 7 simulation export when a dashboard simulation is run
- Phase 8 chronological drift CSV/JSON and five drift figures

All adversarial test samples remain evaluation-only, and the Streamlit dashboard does not retrain models.